[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage6_langgraph/Stage6_LangGraph.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage6_langgraph/Stage6_LangGraph.ipynb

# 🕸️ Stage 6 — LangGraph Workflow
**CodePilot AI Studio | Module 4 | Homework**

---

## What You Will Learn
- Why linear chains are **not enough** for complex agents
- What a **StateGraph** is — nodes, edges, shared state
- How **conditional edges** create dynamic branching and loops
- How to simulate **multi-agent conversation** (AutoGen concept)

## LangChain vs LangGraph
```
LangChain (linear):        LangGraph (dynamic):
A → B → C → END           A → B → C
                                   ↓
Always same path.          approved? YES → END
Cannot loop back.                    NO  → back to A

                           Path decided at RUNTIME!
```

⏱ **Expected time: 30 minutes**

## Step 1 — Paste Your Groq API Key
**Get your free key at https://console.groq.com → API Keys → Create API Key**

Steps:
1. Go to **https://console.groq.com**
2. Sign up free (Google account or email — no credit card)
3. Click **API Keys** in the left sidebar
4. Click **Create API Key** → name it anything → click Submit
5. **Copy the key** (looks like `gsk_xxxx...`) — save it in Notepad
6. Paste it below between the quotes

> Each student must use their own key. Never share your key.

In [ ]:
# ── GROQ API KEY ─────────────────────────────────────────────
# Paste your key between the quotes below.
# Get it free from: https://console.groq.com → API Keys → Create API Key

GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace this!

# Set as environment variable so LangChain can find it automatically
import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Quick check
if GROQ_API_KEY == "paste-your-groq-key-here" or len(GROQ_API_KEY) < 20:
    print("ERROR: Please paste your real Groq API key above!")
    print("Get it from: https://console.groq.com → API Keys")
else:
    print(f"API key accepted! Starts with: {GROQ_API_KEY[:8]}...")
    print("Ready to proceed to the next cell.")

## Step 2 — Setup

In [ ]:
print("Installing packages...")
!pip install -q langchain-groq langchain langchain-community langgraph
print("Setup complete! LangGraph installed.")

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 6: LangGraph Workflow
# ============================================================

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from typing import TypedDict

llm = ChatGroq(model="llama3-8b-8192", temperature=0.3)

# ── STEP 1: Define the Shared State ──────────────────────────
# State is a shared 'whiteboard' all nodes read from and write to.
# Every node gets the full state as input and returns updates.
class AgentState(TypedDict):
    code:         str    # the buggy code (never changes)
    analysis:     str    # written by analyze_node
    fix:          str    # written by fix_node
    critique:     str    # written by reflect_node
    iteration:    int    # incremented by fix_node
    approved:     bool   # set by reflect_node → controls routing
    final_output: str    # the final answer returned to user

# ── STEP 2: Define each node function ─────────────────────────
# Each node is a plain Python function.
# It receives state and returns a DICT of fields to update.

def analyze_node(state: AgentState) -> dict:
    """Analyze ALL bugs in the code."""
    print(f"  [Node: ANALYZE] iteration {state['iteration'] + 1}")
    result = llm.invoke(
        f"List ALL bugs in this Python code. Be specific.\n"
        f"```python\n{state['code']}\n```"
    ).content
    return {"analysis": result}   # only return changed fields!

def fix_node(state: AgentState) -> dict:
    """Generate a fix based on the analysis."""
    print("  [Node: FIX] generating fix...")
    result = llm.invoke(
        f"Fix ALL these bugs:\nCode:\n```python\n{state['code']}\n```\n"
        f"Analysis:\n{state['analysis']}\n"
        f"Show complete fixed code with inline comments."
    ).content
    return {"fix": result, "iteration": state["iteration"] + 1}

def reflect_node(state: AgentState) -> dict:
    """Critique the fix. Set approved=True to stop, False to loop."""
    print("  [Node: REFLECT] critiquing fix...")
    result = llm.invoke(
        f"Does this fix solve ALL bugs? Write APPROVED or NEEDS_IMPROVEMENT.\n"
        f"Original:\n```python\n{state['code']}\n```\nFix:\n{state['fix']}"
    ).content

    is_approved = ("APPROVED" in result.upper()
                   and "NEEDS_IMPROVEMENT" not in result.upper())

    final = f"Fix (iteration {state['iteration']}):\n{state['fix']}" if is_approved else ""
    print(f"  Result: {'APPROVED' if is_approved else 'NEEDS_IMPROVEMENT'}")
    return {"critique": result, "approved": is_approved, "final_output": final}

# ── STEP 3: Conditional routing function ──────────────────────
# This function reads state and returns the NEXT NODE NAME as a string.
# LangGraph uses this to decide where to go after reflect_node.
def should_continue(state: AgentState) -> str:
    MAX_ITER = 2
    if state["approved"]:
        return "end"      # stop — fix is approved!
    elif state["iteration"] >= MAX_ITER:
        return "end"      # stop — hit max iterations safety limit
    else:
        return "analyze"  # loop back — try again!

# ── STEP 4: Build the graph ───────────────────────────────────
graph = StateGraph(AgentState)          # create graph with our state schema
graph.add_node("analyze", analyze_node) # add nodes (name → function)
graph.add_node("fix",     fix_node)
graph.add_node("reflect", reflect_node)

graph.add_edge("analyze", "fix")        # fixed edges: always go here
graph.add_edge("fix", "reflect")
graph.add_conditional_edges(            # dynamic edge: decided at runtime!
    "reflect",
    should_continue,
    {"analyze": "analyze", "end": END}  # maps return value → next node
)
graph.set_entry_point("analyze")        # start here
app = graph.compile()                   # validate and prepare

# ── STEP 5: Run the graph ─────────────────────────────────────
buggy_code = """
def divide_all(numbers, divisor):
    results = []
    for num in numbers:
        results.append(num / divisor)
    return results

print(divide_all([10, 20, 30], 2))
print(divide_all([10, 20, 30], 0))   # ZeroDivisionError!
"""

initial_state = {
    "code": buggy_code, "analysis": "", "fix": "",
    "critique": "", "iteration": 0, "approved": False, "final_output": ""
}

print("=" * 60)
print("CodePilot AI Studio — Stage 6: LangGraph Workflow")
print("=" * 60)
print("Running graph... (watch nodes execute in order below)")
print()

# stream() runs the graph and yields state after each node
final_state = None
for step in app.stream(initial_state):
    node_name = list(step.keys())[0]
    final_state = step[node_name]
    print(f"  Completed node: {node_name} | iteration: {final_state.get('iteration',0)}")

print()
print("FINAL OUTPUT:")
print(final_state.get("final_output") or final_state.get("fix", ""))
print()
print("KEY: The graph decided its own path based on the critique result!")

## Step 4 — Multi-Agent (AutoGen Concept)
LangGraph handled ONE agent with a loop. Now let us simulate TWO agents talking to each other.

In [ ]:
# ── MULTI-AGENT SIMULATION (AutoGen Concept) ─────────────────
# AutoGen runs multiple agents that converse with each other.
# Each agent has ONE specialised role.
# They share outputs as messages — like a WhatsApp chat between AIs.

def multi_agent_review(code: str) -> dict:
    """Two agents: DebuggerBot writes a fix, ReviewerBot critiques it."""

    print("[DebuggerBot] Generating fix...")
    debugger_response = llm.invoke(
        f"You are DebuggerBot. Find and fix the bug in this code.\n"
        f"```python\n{code}\n```\nProvide the fix with brief comments."
    ).content

    print("[ReviewerBot] Reviewing DebuggerBot's fix...")
    reviewer_response = llm.invoke(
        f"You are ReviewerBot. Review this fix.\n"
        f"Original: ```python\n{code}\n```\n"
        f"DebuggerBot's fix:\n{debugger_response}\n\n"
        f"Write: APPROVED with reason, or REJECTED with what to fix."
    ).content

    return {"debugger": debugger_response, "reviewer": reviewer_response}

print("=" * 60)
print("Multi-Agent Review (AutoGen Concept)")
print("=" * 60)

result = multi_agent_review(buggy_code)
print()
print("=== DebuggerBot output ===")
print(result["debugger"])
print()
print("=== ReviewerBot verdict ===")
print(result["reviewer"])
print()
print("KEY: Each agent has ONE specialised prompt.")
print("They share outputs as messages — same concept as AutoGen!")

## Step 5 — Experiments

In [ ]:
# ── EXPERIMENT: Add a documentation node ──────────────────────
# Extend the graph with a new node after fix_node.
# This node writes a docstring for the fixed function.

def document_node(state: AgentState) -> dict:
    """Write a docstring for the fixed code."""
    print("  [Node: DOCUMENT] writing docstring...")
    result = llm.invoke(
        f"Write a proper Python docstring for this fixed function.\n"
        f"Include: what it does, parameters, return value, raises.\n"
        f"Fixed code:\n{state['fix']}"
    ).content
    # Append docstring to final output
    return {"final_output": result + "\n\n" + state.get("final_output", "")}

# Build extended graph
graph2 = StateGraph(AgentState)
graph2.add_node("analyze",  analyze_node)
graph2.add_node("fix",      fix_node)
graph2.add_node("reflect",  reflect_node)
graph2.add_node("document", document_node)  # NEW NODE!

graph2.add_edge("analyze", "fix")
graph2.add_edge("fix", "reflect")
graph2.add_conditional_edges(
    "reflect", should_continue,
    {"analyze": "analyze", "end": "document"}  # go to document instead of END!
)
graph2.add_edge("document", END)  # document → END
graph2.set_entry_point("analyze")
app2 = graph2.compile()

print("Running extended graph with document node...")
final2 = None
for step in app2.stream(initial_state):
    node_name = list(step.keys())[0]
    final2 = step[node_name]
    print(f"  Completed: {node_name}")

print("\nFinal output with documentation:")
print(final2.get("final_output", ""))

## Summary — Stage 6 Complete!

| Concept | What it means |
|---------|---------------|
| **StateGraph** | Graph where nodes share a typed state |
| **Node** | Python function that reads/writes state |
| **Edge** | Fixed connection between two nodes |
| **Conditional edge** | Dynamic routing based on state value |
| **`should_continue()`** | Returns next node name as string |
| **`app.stream()`** | Runs graph, yields state after each node |
| **Multi-agent** | Multiple specialised agents passing messages |

---
## Congratulations! All 6 Stages Complete!

You have built a complete CodePilot AI Studio with:
- Stage 1: Cloud LLM with Groq
- Stage 2: Short-term memory
- Stage 3: Tool orchestration
- Stage 4: RAG long-term memory
- Stage 5: Reflection and self-improvement
- Stage 6: LangGraph stateful workflow + multi-agent